In [2]:
import duckdb
import polars as pl

db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"

# Read-only rejimində qoşuluruq
con = duckdb.connect(db_path, read_only=True)

print("=== 1. WAIT TIME VS. TIP ANALYSIS ===")

query_wait_vs_tip = """
    SELECT 
        CASE 
            WHEN epoch(pickup_datetime - request_datetime) / 60.0 < 2 THEN '1. Under 2 min'
            WHEN epoch(pickup_datetime - request_datetime) / 60.0 BETWEEN 2 AND 5 THEN '2. 2-5 min'
            WHEN epoch(pickup_datetime - request_datetime) / 60.0 BETWEEN 5 AND 10 THEN '3. 5-10 min'
            ELSE '4. Over 10 min'
        END AS wait_time_bucket,
        COUNT(*) AS total_trips,
        ROUND(AVG(tips), 2) AS avg_tip_amount,
        ROUND(SUM(CASE WHEN tips > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS tip_percentage
    FROM fact_trips
    GROUP BY wait_time_bucket
    ORDER BY wait_time_bucket;
"""

df_wait_tip = con.sql(query_wait_vs_tip).pl()
print(df_wait_tip)

con.close()

=== 1. WAIT TIME VS. TIP ANALYSIS ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

shape: (4, 4)
┌──────────────────┬─────────────┬────────────────┬────────────────┐
│ wait_time_bucket ┆ total_trips ┆ avg_tip_amount ┆ tip_percentage │
│ ---              ┆ ---         ┆ ---            ┆ ---            │
│ str              ┆ i64         ┆ f64            ┆ f64            │
╞══════════════════╪═════════════╪════════════════╪════════════════╡
│ 1. Under 2 min   ┆ 27634391    ┆ 1.64           ┆ 24.21          │
│ 2. 2-5 min       ┆ 132255064   ┆ 1.11           ┆ 18.9           │
│ 3. 5-10 min      ┆ 68756510    ┆ 1.14           ┆ 17.23          │
│ 4. Over 10 min   ┆ 14943719    ┆ 1.37           ┆ 17.27          │
└──────────────────┴─────────────┴────────────────┴────────────────┘


In [3]:
import duckdb
import polars as pl

db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"
con = duckdb.connect(db_path, read_only=True)

print("=== 2. AIRPORT TRIP PROFITABILITY ANALYSIS ===")

query_airports = """
    SELECT 
        company_id,
        CASE 
            WHEN PULocationID IN (132, 138) THEN 'Airport Pickup'
            WHEN DOLocationID IN (132, 138) THEN 'Airport Dropoff'
            ELSE 'Standard Trip'
        END AS airport_trip_type,
        COUNT(*) AS total_trips,
        ROUND(AVG(total_gross_revenue), 2) AS avg_gross_revenue,
        ROUND(AVG(company_margin), 2) AS avg_company_margin,
        ROUND(AVG(driver_pay), 2) AS avg_driver_pay
    FROM fact_trips
    WHERE PULocationID IN (132, 138) OR DOLocationID IN (132, 138)
    GROUP BY company_id, airport_trip_type
    ORDER BY total_trips DESC;
"""

df_airports = con.sql(query_airports).pl()
print(df_airports)

con.close()

=== 2. AIRPORT TRIP PROFITABILITY ANALYSIS ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

shape: (4, 6)
┌────────────┬───────────────────┬─────────────┬───────────────────┬────────────────────┬────────────────┐
│ company_id ┆ airport_trip_type ┆ total_trips ┆ avg_gross_revenue ┆ avg_company_margin ┆ avg_driver_pay │
│ ---        ┆ ---               ┆ ---         ┆ ---               ┆ ---                ┆ ---            │
│ str        ┆ str               ┆ i64         ┆ f64               ┆ f64                ┆ f64            │
╞════════════╪═══════════════════╪═════════════╪═══════════════════╪════════════════════╪════════════════╡
│ HV0003     ┆ Airport Dropoff   ┆ 7511410     ┆ 70.11             ┆ 19.01              ┆ 40.66          │
│ HV0003     ┆ Airport Pickup    ┆ 6515668     ┆ 80.7              ┆ 18.38              ┆ 49.67          │
│ HV0005     ┆ Airport Dropoff   ┆ 2744378     ┆ 62.49             ┆ 13.6               ┆ 38.67          │
│ HV0005     ┆ Airport Pickup    ┆ 2476365     ┆ 71.92             ┆ 11.81              ┆ 48.09          │
└────────────┴─────────

In [4]:
# Query 3: Driver Earnings & Efficiency Analysis
import duckdb
import polars as pl

db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"
con = duckdb.connect(db_path, read_only=True)

print("=== 3. DRIVER EARNINGS & EFFICIENCY ANALYSIS ===")

query_driver_efficiency = """
    SELECT 
        HOUR(pickup_datetime) AS pickup_hour,
        COUNT(*) AS total_trips,
        ROUND(AVG(driver_pay), 2) AS avg_driver_pay,
        ROUND(SUM(driver_pay) / (SUM(epoch(dropoff_datetime - pickup_datetime)) / 3600.0), 2) AS driver_pay_per_hour_worked,
        ROUND(SUM(driver_pay) / SUM(trip_miles), 2) AS driver_pay_per_mile
    FROM fact_trips
    GROUP BY pickup_hour
    ORDER BY driver_pay_per_hour_worked DESC;
"""

df_driver_eff = con.sql(query_driver_efficiency).pl()
print(df_driver_eff)

con.close()

=== 3. DRIVER EARNINGS & EFFICIENCY ANALYSIS ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

shape: (24, 5)
┌─────────────┬─────────────┬────────────────┬────────────────────────────┬─────────────────────┐
│ pickup_hour ┆ total_trips ┆ avg_driver_pay ┆ driver_pay_per_hour_worked ┆ driver_pay_per_mile │
│ ---         ┆ ---         ┆ ---            ┆ ---                        ┆ ---                 │
│ i64         ┆ i64         ┆ f64            ┆ f64                        ┆ f64                 │
╞═════════════╪═════════════╪════════════════╪════════════════════════════╪═════════════════════╡
│ 4           ┆ 3743159     ┆ 23.01          ┆ 76.16                      ┆ 3.27                │
│ 5           ┆ 4446709     ┆ 23.13          ┆ 73.76                      ┆ 3.23                │
│ 3           ┆ 3540262     ┆ 20.48          ┆ 71.46                      ┆ 3.38                │
│ 2           ┆ 4367605     ┆ 19.27          ┆ 69.34                      ┆ 3.59                │
│ 1           ┆ 6098806     ┆ 19.3           ┆ 68.62                      ┆ 3.68                │
│ …  

In [6]:
# Query 4: Borough Market Share Dispersion
import duckdb
import polars as pl

db_path = r"C:\Users\ASUS\Desktop\nyc\NYC_HVFHV_2025\nyc_ride_hailing_2025.duckdb"
con = duckdb.connect(db_path, read_only=True)

print("=== 4. BOROUGH MARKET SHARE DISPERSION ===")

query_borough_share = """
    SELECT 
        loc.borough,
        fact.company_id,
        COUNT(*) AS total_trips,
        ROUND(SUM(fact.total_gross_revenue), 2) AS total_gross_revenue,
        ROUND(AVG(fact.trip_miles), 2) AS avg_trip_miles
    FROM fact_trips fact
    JOIN dim_location loc ON fact.PULocationID = loc.location_id
    GROUP BY loc.borough, fact.company_id
    ORDER BY loc.borough, total_trips DESC;
"""

df_borough_share = con.sql(query_borough_share).pl()
print(df_borough_share)

con.close()

=== 4. BOROUGH MARKET SHARE DISPERSION ===
shape: (14, 5)
┌───────────────┬────────────┬─────────────┬─────────────────────┬────────────────┐
│ borough       ┆ company_id ┆ total_trips ┆ total_gross_revenue ┆ avg_trip_miles │
│ ---           ┆ ---        ┆ ---         ┆ ---                 ┆ ---            │
│ str           ┆ str        ┆ i64         ┆ f64                 ┆ f64            │
╞═══════════════╪════════════╪═════════════╪═════════════════════╪════════════════╡
│ Bronx         ┆ HV0003     ┆ 23680225    ┆ 4.9118e8            ┆ 4.4            │
│ Bronx         ┆ HV0005     ┆ 7725806     ┆ 1.6295e8            ┆ 4.14           │
│ Brooklyn      ┆ HV0003     ┆ 46402943    ┆ 1.1354e9            ┆ 4.23           │
│ Brooklyn      ┆ HV0005     ┆ 18909401    ┆ 4.5954e8            ┆ 4.01           │
│ EWR           ┆ HV0003     ┆ 14          ┆ 711.6               ┆ 3.81           │
│ …             ┆ …          ┆ …           ┆ …                   ┆ …              │
│ Queens        ┆ 